In [1]:
from utils.util_octree_stuff import *
from utils.util_sample_stuff import *
from utils.util_visual_stuff import *
from loss.octree_losses import *
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from dataset.voxel_dataset import VoxelPatchDataset, get_dataloader

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
DEPTH_IN = 6
DEPTH_STOP = 6 # can try 5 
LATENT_DIM = 16
FULL_DEPTH = 3
TOTAL_SEMANTIC_CLASSES = 92

PATCH_DIR = "data/patches_128x128x64_s16"
loader = get_dataloader(PATCH_DIR, batch_size=1)
from models.networks.dualoctree_networks.graph_sem_vae import GraphVAE

vae = GraphVAE(
    depth=DEPTH_IN,
    channel_in=32,     
    nout=2,
    full_depth=FULL_DEPTH,
    depth_stop=DEPTH_STOP ,
    depth_out=DEPTH_IN,
    latent_dim=LATENT_DIM,  
    num_classes=TOTAL_SEMANTIC_CLASSES,# latent dimension
    resblk_num=2,
    
).cuda()

vae.load_state_dict(torch.load("saved_model/vae_model_4.pt"),vae.parameters())

vae.train()

/tmp/ipykernel_78018/1344920701.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vae.load_state_dict(torch.load("saved_model/vae_model_4.pt"),vae.parameters())


GraphVAE(
  (patch_embed): SharedPatchEmbed(
    (embedding): Embedding(92, 32)
    (conv_proj): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (3): ReLU()
      (4): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (5): ReLU()
    )
    (to_latent): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=256, out_features=32, bias=True)
    )
  )
  (conv1): GraphConv(channel_in=32, channel_out=32, n_edge_type=7, avg_degree=7, n_node_type=5)
  (encoder): ModuleList(
    (0): GraphResBlocks(
      (resblks): ModuleList(
        (0): GraphResBlock(
          (norm1): DualOctreeGroupNorm(in_channels=32, group=8, nempty=False)
          (conv1): GraphConv(channel_in=32, channel_out=32, n_edge_type=7, avg_degree=7, n_node_type=5)
          (norm2): DualOctreeGroupNorm(in_channels=32, group=8, ne

In [3]:
import tqdm
from models.networks.dualoctree_networks.graph_sem_vae import GraphVAE

T_max = 100

optimizer = torch.optim.Adam(vae.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max, eta_min=1e-7)
for epoch in range(100):
    pbar = tqdm.tqdm(loader, desc=f"Epoch {epoch}")
    vae.train()
    
    for vox, _ in pbar:
        vox = vox.cuda().long()
        
        # 1. Create Octree directly from voxel occupancy
        # (Assuming your points2octree logic just uses non-empty voxel locations)
        # If possible, move this logic into your Dataset class to speed up training!
        non_empty_mask = get_non_empty_mask(vox[0], 2)
        points = voxel_grid_to_points(non_empty_mask)
        octree = points2octree(points, depth=DEPTH_IN, full_depth=FULL_DEPTH).cuda()
        
        # 2. Forward Pass
        # Pass features directly if your model supports it
        patch_feat = voxel_to_patch(vox, patch_size=2)
        assign_octree_patch_features(patch_feat[0], octree, DEPTH_IN)
        
        output = vae(octree, octree_out=octree, update_octree = False)
        
        # 3. Compute Losses
        # Use the 'clean' semantic loss we just built
        sem_loss_dict = compute_semantic_loss(output['sem_voxs'], output['octree_out'], vox, torch.ones(TOTAL_SEMANTIC_CLASSES).cuda())
        
        oct_loss_dict = compute_octree_loss(output['logits'], octree)
        
  
        
        # Combine all losses efficiently
        total_loss = output['kl_loss'] + sem_loss_dict["sem_loss_6"]
        total_loss += sum(v for k, v in oct_loss_dict.items() if 'loss_' in k)

        # 4. Optimization
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

    
        pbar.set_postfix({
            "loss": f"{total_loss.item():.3f}",
            "sem": f"{sem_loss_dict['sem_loss_6'].item():.3f}",
            "acc": f"{sem_loss_dict['sem_accu_6'].item():.3f}"
        })
        break
    break

    scheduler.step()
    torch.save(vae.state_dict(), f"saved_model/vae_model_{epoch}.pt")

Epoch 0:   0%|          | 0/281 [00:00<?, ?it/s]/home/zxj/Desktop/Octree-Scene-Diffusion/dataset/voxel_dataset.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = to

In [4]:
recon_voxel = reconstruct_voxel_from_patch(
    sem_voxs=output['sem_voxs'],
    octree=output['octree_out'],
    depth=6,
    shape=(1, 128, 128, 64)
)
visualize_voxel_labels(recon_voxel[0], voxel_size=0.05, origin=(0,0,0))

Visualizing voxel grid: shape=(128, 128, 64), filled=56528


In [3]:

from models.networks.diffusion_networks.graph_unet_hr import UNet3DModel

# === 1. Hyperparams ===
in_channels = LATENT_DIM        # dimension of latent z
model_channels = 64

lr_model_channels = 128
out_channels = LATENT_DIM      # same as input if predicting noise
image_size = 16
depth = DEPTH_STOP
full_depth = FULL_DEPTH
channel_mult = [1,2]
num_res_blocks = [1,1, 1]





# === 4. Init and forward model ===
unet_model = UNet3DModel(
    image_size=image_size,
    input_depth=depth,
    full_depth=full_depth,
    in_channels=in_channels,
    model_channels=model_channels,
    lr_model_channels=lr_model_channels,
    out_channels=out_channels,
    num_res_blocks=num_res_blocks,
    dropout=0.1,
    channel_mult=channel_mult,
    use_checkpoint=False,
).cuda()


In [ ]:

import matplotlib.pyplot as plt
from IPython.display import clear_output

from tqdm.auto import tqdm
optimizer = torch.optim.AdamW(unet_model.parameters(), lr=1e-4,weight_decay=0)
T_MAX = 600
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX, eta_min=1e-6)



loss_history = []

for epoch in range(0, T_MAX, 1):
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for vox, _ in pbar:
        vox = vox.type(torch.LongTensor).cuda()

        # === preprocess voxel/octree as before ===
        patch = voxel_to_patch(vox, patch_size=2)
        non_empty_mask = get_non_empty_mask(vox[0], 2)
        points = voxel_grid_to_points(non_empty_mask)
        octree = points2octree(points, depth=6, full_depth=FULL_DEPTH)
        assign_octree_patch_features(patch[0], octree, 6)
        octree = octree.cuda()

        z, mu, doctree = vae.extract_code(octree)
        z = mu.cuda()

        # === Continuous timestep sampling ===
        t_cont = torch.rand(1, device=z.device)
        logsnr_t = log_snr_schedule_cosine(t_cont)
        alpha_t, sigma_t = log_snr_to_alpha_sigma(logsnr_t)

        noise = torch.randn_like(z)
        z_t = alpha_t * z + sigma_t * noise

        # === UNet forward ===
        pred_x0 = unet_model(z_t, doctree=doctree, timesteps=t_cont)
        loss = F.mse_loss(pred_x0, z)
    #     break
    # break
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet_model.parameters(), 1.0)
        optimizer.step()

        # === Track and plot loss ===
        loss_history.append(loss.item())
        clear_output(wait=True)
        plt.figure(figsize=(8, 4))
        plt.plot(loss_history, label="Loss")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("Training Loss Curve")
        plt.legend()
        plt.grid(True)
        plt.show()

        pbar.set_postfix({
            "t": f"{t_cont.item():.4f}",
            "loss": f"{loss.item():.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.4e}"
        })
    lr_scheduler.step()

In [7]:
torch.save(unet_model.state_dict(), 'saved_model/unet.pt')

In [8]:
z_rand = torch.randn(doctree.total_num, LATENT_DIM)
z_T = z_rand.cuda()
z_sampled = ddim_sample_logsnr_cosine(
    z_T=z_T,
    model=unet_model,
    doctree=doctree,
    S=40
    
)
output = vae.decode_code(z_sampled.cuda(), doctree, update_octree=False, pos=None)
recon_voxel = reconstruct_voxel_from_patch(
    sem_voxs=output['sem_voxs'],
    octree=output['octree_out'],
    depth=6,
    shape=(1, 128, 128, 64)
)
visualize_voxel_labels(recon_voxel[0], voxel_size=0.05, origin=(0,0,0))

Visualizing voxel grid: shape=(128, 128, 64), filled=68170
